# Bias in Bios Concept EDA

This notebook designs the concept set for the `LabHC/bias_in_bios` STAQ experiment.

The goal is simple: inspect the biography data, identify profession-relevant evidence cues, and produce one clean query vocabulary before training any STAQ model.

Working target:

- input: short biography text
- prediction target: 28-way profession classification
- sample-level sensitive variable: binary gender label provided by the dataset
- query vocabulary: about 40 human-readable biography concepts
- sensitive set: the subset of query concepts that directly expose, or strongly proxy, gender

Important distinction: the dataset's `gender` column is the protected sample-level variable. The STAQ sensitive set is a subset of the query vocabulary, such as pronouns, gendered titles, names, or family-role concepts.

In [ ]:
%pip install -e ..
%pip install datasets -q

In [1]:
%load_ext autoreload
%autoreload 2

from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset

from staq.config import default_paths, detect_device
from staq.training import seed_everything

/opt/miniconda3/envs/vip_conceptqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_root = Path.cwd().resolve()
if not (repo_root / "staq").exists() and (repo_root.parent / "staq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

device = detect_device(allow_mps=False)
random_seed = 0
seed_everything(random_seed)

dataset_name = "LabHC/bias_in_bios"
text_column = "hard_text"
target_column = "profession"
sensitive_column = "gender"

profession_names = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher",
]

gender_names = {0: "male", 1: "female"}

{
    "repo_root": str(repo_root),
    "device": str(device),
    "dataset_name": dataset_name,
    "num_professions": len(profession_names),
}

{'repo_root': '/Users/amir.atashin/Projects/conceptqa_vip-main',
 'device': 'cpu',
 'dataset_name': 'LabHC/bias_in_bios',
 'num_professions': 28}

In [3]:
raw_train = load_dataset(dataset_name, split="train")
raw_dev = load_dataset(dataset_name, split="dev")
raw_test = load_dataset(dataset_name, split="test")

{
    "train": len(raw_train),
    "dev": len(raw_dev),
    "test": len(raw_test),
    "columns": raw_train.column_names,
}

{'train': 257478,
 'dev': 39642,
 'test': 99069,
 'columns': ['hard_text', 'profession', 'gender']}

In [4]:
def label_distribution(dataset, column, names=None):
    counts = Counter(dataset[column])
    total = len(dataset)
    rows = []
    for label, count in sorted(counts.items(), key=lambda item: item[0]):
        rows.append(
            {
                "label": int(label),
                "name": names[label] if names is not None else str(label),
                "count": int(count),
                "share": count / total,
            }
        )
    return pd.DataFrame(rows)

profession_dist = label_distribution(raw_train, target_column, profession_names)
gender_dist = label_distribution(raw_train, sensitive_column, gender_names)

display(gender_dist)
display(profession_dist.sort_values("share", ascending=False).head(10))

,label,name,count,share
0,0,male,138780,0.538998
1,1,female,118698,0.461002


,label,name,count,share
21,21,professor,76748,0.298076
19,19,physician,26648,0.103496
2,2,attorney,21169,0.082217
18,18,photographer,15773,0.061260
11,11,journalist,12960,0.050334
13,13,nurse,12316,0.047833
22,22,psychologist,11945,0.046392
26,26,teacher,10531,0.040901
6,6,dentist,9479,0.036815
25,25,surgeon,8829,0.034290


In [5]:
for idx in range(3):
    row = raw_train[idx]
    print("-" * 80)
    print("profession:", profession_names[row[target_column]])
    print("gender:", gender_names[row[sensitive_column]])
    print(row[text_column][:900])

--------------------------------------------------------------------------------
profession: professor
gender: male
He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates
--------------------------------------------------------------------------------
profession: nurse
gender: female
She is able to assess, diagnose and treat minor illness conditions and exacerbations of some long term conditions. Her qualifications include Registered General Nurse, Bachelor of Nursing, Diploma in Health Science, Emergency Care Practitioner and Independent Nurse Prescribing.
--------------------------------------------------------------------------------
profession: attorney
gender: female
Prior to law school, Brittni graduated magna cum laude from DePaul University in 2011 with her Bachelor’s Degree in Psychol

## Concept Set Design Criteria

Before training any STAQ model, we first fix the query vocabulary. A good target is about 40 concepts: large enough to cover the profession task, but small enough that every concept can be inspected and justified.

The concept set should balance:

- utility concepts: profession-relevant evidence cues, not profession names
- direct sensitive concepts: explicit gender cues, such as pronouns or gendered titles
- proxy-sensitive concepts: cues that may reveal gender indirectly, such as family-role mentions or strongly gendered biography patterns

The concepts should be useful but not too revealing. Ideally, an active-query agent should need several questions to reach high confidence, rather than solving the task from one obvious shortcut.

## Data-Driven Profession Cues

Next, we inspect words that are unusually common for each profession compared with the rest of the dataset.

The purpose is not to copy raw words into the concept set. The purpose is to translate raw biography evidence into broader, human-readable concepts. For example, terms like `vinyasa`, `asana`, and `pranayama` suggest a broader concept such as `mind_body_practice`.

In [9]:
import re

stopwords = {
    "a", "about", "after", "also", "an", "and", "are", "as", "at", "be", "been", "by",
    "can", "for", "from", "has", "have", "he", "her", "his", "in", "into", "is", "it",
    "its", "of", "on", "or", "she", "that", "the", "their", "they", "this", "to", "was",
    "were", "who", "with", "within", "years", "year", "new", "work", "works", "working",
    "professional", "currently", "including", "include", "based", "born", "known", "became",
}


def tokenize(text):
    tokens = re.findall(r"[a-z][a-z]+", text.lower())
    return [token for token in tokens if token not in stopwords and len(token) >= 3]


def profession_token_counts(dataset, max_rows=50000):
    selected = dataset.select(range(min(max_rows, len(dataset))))
    by_profession = {name: Counter() for name in profession_names}
    global_counts = Counter()
    profession_sizes = Counter()

    for row in selected:
        profession = profession_names[row[target_column]]
        tokens = tokenize(row[text_column])
        by_profession[profession].update(tokens)
        global_counts.update(tokens)
        profession_sizes[profession] += 1

    return by_profession, global_counts, profession_sizes


by_profession_counts, global_token_counts, profession_sample_sizes = profession_token_counts(raw_train)

{
    "sample_size": sum(profession_sample_sizes.values()),
    "num_professions": len(profession_sample_sizes),
    "vocabulary_size": len(global_token_counts),
}

{'sample_size': 50000, 'num_professions': 28, 'vocabulary_size': 87806}

In [10]:
def distinctive_terms(profession, min_count=20, top_k=15, smoothing=5.0):
    profession_counts = by_profession_counts[profession]
    profession_total = sum(profession_counts.values())
    global_total = sum(global_token_counts.values())

    rows = []
    for token, count in profession_counts.items():
        if count < min_count:
            continue
        other_count = global_token_counts[token] - count
        other_total = global_total - profession_total
        profession_rate = (count + smoothing) / (profession_total + smoothing)
        other_rate = (other_count + smoothing) / (other_total + smoothing)
        rows.append(
            {
                "term": token,
                "count_in_profession": int(count),
                "count_overall": int(global_token_counts[token]),
                "profession_rate": profession_rate,
                "rest_rate": other_rate,
                "rate_ratio": profession_rate / other_rate,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(["rate_ratio", "count_in_profession"], ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )


distinctive_terms("software_engineer", top_k=15)

,term,count_in_profession,count_overall,profession_rate,rest_rate,rate_ratio
0,websphere,37,46,0.001475,0.000008,189.126400
1,apache,63,82,0.002388,0.000013,178.619378
2,python,28,40,0.001159,0.000009,122.375906
3,linux,40,61,0.001580,0.000014,109.111385
4,javascript,34,52,0.001369,0.000013,106.897530
5,hadoop,23,38,0.000983,0.000011,88.258987
6,android,23,38,0.000983,0.000011,88.258987
7,agile,28,49,0.001159,0.000014,80.015015
8,server,57,103,0.002177,0.000028,76.639456
9,java,84,161,0.003125,0.000046,68.423779


In [11]:
def show_profession_terms(profession, top_k=15, min_count=20):
    return distinctive_terms(
        profession=profession,
        min_count=min_count,
        top_k=top_k,
    )[["term", "count_in_profession", "count_overall", "rate_ratio"]]


show_profession_terms("yoga_teacher", top_k=20, min_count=5)

,term,count_in_profession,count_overall,rate_ratio
0,vinyasa,24,29,701.716172
1,hatha,12,15,514.188574
2,anusara,5,7,345.672991
3,asana,5,7,345.672991
4,yoga,435,741,342.338525
5,pranayama,6,9,332.710254
6,ashtanga,11,19,297.810577
7,yin,7,12,290.365312
8,iyengar,9,16,282.299609
9,kripalu,5,10,241.971094


In [12]:
profession_sample_summary = pd.DataFrame(
    {
        "profession": profession_names,
        "examples_in_token_sample": [int(profession_sample_sizes[name]) for name in profession_names],
    }
)

profession_sample_summary

,profession,examples_in_token_sample
0,accountant,723
1,architect,1232
2,attorney,4011
3,chiropractor,334
4,comedian,351
5,composer,700
6,dentist,1832
7,dietitian,459
8,dj,182
9,filmmaker,896


This section supports concept design with one simple check: `show_profession_terms("profession_name")` shows raw words that are unusually common for a profession.

When a raw word is just the class name, it is useful evidence but usually not a good concept name. The concept set should use broader evidence cues rather than direct profession labels.

## Profession Cue Summary

We summarize the data-driven terms for all 28 professions in one table.

The `distinctive_terms` column contains raw dataset words. The `concept_hint` column translates those words into broader evidence cues. This table is the bridge from dataset inspection to the proposed STAQ concept set.

In [13]:
def top_terms_text(profession, top_k=8, min_count=10):
    terms = show_profession_terms(
        profession=profession,
        top_k=top_k,
        min_count=min_count,
    )["term"].tolist()
    return ", ".join(terms)


profession_concept_hints = {
    "accountant": "financial records, auditing, tax work",
    "architect": "building design, spatial design, construction projects",
    "attorney": "legal practice, court work, law training",
    "chiropractor": "manual therapy, musculoskeletal care, clinic work",
    "comedian": "comedy performance, stage work, entertainment",
    "composer": "music composition, musical performance, scores",
    "dentist": "dental care, oral health, clinical treatment",
    "dietitian": "nutrition counseling, diet planning, health coaching",
    "dj": "music performance, clubs, radio, live events",
    "filmmaker": "film production, directing, documentary work",
    "interior_designer": "interior spaces, decoration, design projects",
    "journalist": "reporting, news writing, editorial work",
    "model": "fashion work, modeling, visual presentation",
    "nurse": "patient care, nursing credentials, clinical care",
    "painter": "visual art, painting, galleries, exhibitions",
    "paralegal": "legal support, case preparation, law office work",
    "pastor": "ministry, church leadership, religious service",
    "personal_trainer": "fitness instruction, exercise coaching, physical training",
    "photographer": "camera work, photography, visual media",
    "physician": "medical diagnosis, patient treatment, clinical practice",
    "poet": "poetry, literary writing, publications",
    "professor": "university affiliation, teaching, research publications",
    "psychologist": "therapy, counseling, mental health, psychology training",
    "rapper": "rap music, albums, performance, recording",
    "software_engineer": "programming, software development, technical systems",
    "surgeon": "surgical procedures, operating room, medical specialty",
    "teacher": "classroom teaching, curriculum, student instruction",
    "yoga_teacher": "mind-body practice, movement instruction, wellness training",
}

profession_cue_rows = []
for profession in profession_names:
    profession_cue_rows.append(
        {
            "profession": profession,
            "train_examples_in_sample": int(profession_sample_sizes[profession]),
            "distinctive_terms": top_terms_text(profession, top_k=8, min_count=5),
            "concept_hint": profession_concept_hints[profession],
        }
    )

profession_cue_df = pd.DataFrame(profession_cue_rows)
profession_cue_df

,profession,train_examples_in_sample,distinctive_terms,concept_hint
0,accountant,723,"cpas, payroll, quickbooks, cfo, accountants, c...","financial records, auditing, tax work"
1,architect,1232,"architects, sharepoint, akamai, baidu, aia, em...","building design, spatial design, construction ..."
2,attorney,4011,"disputes, litigated, prosecutor, lawyers, liti...","legal practice, court work, law training"
3,chiropractor,334,"chiropractic, chiropractor, chiropractors, kol...","manual therapy, musculoskeletal care, clinic work"
4,comedian,351,"comedy, comedians, comedian, funniest, funches...","comedy performance, stage work, entertainment"
5,composer,700,"composers, electroacoustic, scoring, orchestra...","music composition, musical performance, scores"
6,dentist,1832,"dentist, crowns, bds, dentures, fixing, lybrat...","dental care, oral health, clinical treatment"
7,dietitian,459,"dietetic, dietitian, dietetics, dietitians, di...","nutrition counseling, diet planning, health co..."
8,dj,182,"remixes, djs, buuren, turntables, djing, techn...","music performance, clubs, radio, live events"
9,filmmaker,896,"screened, filmmaking, sundance, directorial, c...","film production, directing, documentary work"


In [14]:
def profession_examples(dataset, profession, max_rows=20000):
    rows = []
    target_label = profession_names.index(profession)
    selected = dataset.select(range(min(max_rows, len(dataset))))
    for idx, row in enumerate(selected):
        if row[target_column] == target_label:
            rows.append(
                {
                    "idx": idx,
                    "gender": gender_names[row[sensitive_column]],
                    "text": row[text_column],
                }
            )
    return pd.DataFrame(rows)


def inspect_profession(profession, n_examples=5, random_state=0):
    print(f"Profession: {profession}")
    print(f"Examples in 50k-token sample: {profession_sample_sizes[profession]}")
    print("\nDistinctive terms:")
    display(show_profession_terms(profession, top_k=15, min_count=5))
    print("\nExample biographies:")
    examples = profession_examples(raw_train, profession)
    display(
        examples.sample(
            n=min(n_examples, len(examples)),
            random_state=random_state,
        )
    )


inspect_profession("yoga_teacher", n_examples=3)

Profession: yoga_teacher
Examples in 50k-token sample: 195

Distinctive terms:


,term,count_in_profession,count_overall,rate_ratio
0,vinyasa,24,29,701.716172
1,hatha,12,15,514.188574
2,anusara,5,7,345.672991
3,asana,5,7,345.672991
4,yoga,435,741,342.338525
5,pranayama,6,9,332.710254
6,ashtanga,11,19,297.810577
7,yin,7,12,290.365312
8,iyengar,9,16,282.299609
9,kripalu,5,10,241.971094



Example biographies:


,idx,gender,text
2,130,female,Kelli earned her 200 hour Sivananda Teacher Tr...
13,1430,male,His extensive background as a former Anusara t...
53,10824,female,She shares her love of yoga through weekly cla...


Reading rule:

Raw terms are evidence, not final concept names. If several professions share a cue, that cue may be a good reusable concept. If a cue is too narrow, such as a single software package or a specific yoga school, it is grouped into a broader concept.

The proposed concept set below uses this rule to produce about 40 concepts marked as `utility`, `direct_sensitive`, or `proxy_sensitive`.

## Proposed Concept Set

This is the proposed concept set for the BIOS STAQ experiment.

The goal is to create concepts that are useful but not too revealing. A good active-query setup should need several questions to become confident. If a concept almost directly identifies a profession, or if a small pair of concepts makes the answer obvious, then the query vocabulary is too shortcut-heavy.

Design rules:

- prefer biography evidence over profession names
- prefer reusable cues that can appear across multiple professions
- avoid very narrow keywords such as one software package, one yoga school, or one insurance provider
- include a small sensitive set, but keep it clearly separate from utility concepts
- use one lightweight validation pass before freezing the set for STAQ training

In [ ]:
candidate_concepts = [
    {
        "concept": "legal_practice",
        "kind": "utility",
        "description": "Biography mentions legal work, law practice, courts, litigation, or legal representation.",
        "example_terms": ["law", "legal", "court", "litigation", "counsel", "case"],
    },
    {
        "concept": "legal_support_work",
        "kind": "utility",
        "description": "Biography mentions support work around legal cases, documents, filings, or law offices.",
        "example_terms": ["paralegal", "pleadings", "filings", "case preparation", "law office"],
    },
    {
        "concept": "clinical_care",
        "kind": "utility",
        "description": "Biography mentions direct clinical care, diagnosis, treatment, patients, or healthcare settings.",
        "example_terms": ["patient", "clinic", "treatment", "diagnosis", "healthcare"],
    },
    {
        "concept": "medical_training",
        "kind": "utility",
        "description": "Biography mentions medical degrees, residency, fellowship, board certification, or clinical credentials.",
        "example_terms": ["md", "residency", "fellowship", "board certified", "medical school"],
    },
    {
        "concept": "surgical_work",
        "kind": "utility",
        "description": "Biography mentions surgical procedures, operations, operating rooms, or surgical specialties.",
        "example_terms": ["surgery", "surgical", "operation", "procedure", "orthopedic"],
    },
    {
        "concept": "nursing_care",
        "kind": "utility",
        "description": "Biography mentions nursing care, nursing credentials, nurse prokay, niceactitioner work, or bedside care.",
        "example_terms": ["nursing", "registered nurse", "nurse practitioner", "rn", "patient care"],
    },
    {
        "concept": "dental_or_oral_health",
        "kind": "utility",
        "description": "Biography mentions dental care, oral health, teeth, crowns, dentures, or dentistry.",
        "example_terms": ["dental", "dentistry", "oral health", "crowns", "dentures"],
    },
    {
        "concept": "therapy_or_counseling",
        "kind": "utility",
        "description": "Biography mentions therapy, counseling, psychotherapy, mental health, or client sessions.",
        "example_terms": ["therapy", "counseling", "psychotherapy", "mental health", "clients"],
    },
    {
        "concept": "nutrition_or_diet_guidance",
        "kind": "utility",
        "description": "Biography mentions nutrition, diet planning, food guidance, or health coaching around eating.",
        "example_terms": ["nutrition", "diet", "dietetic", "meal planning", "food"],
    },
    {
        "concept": "manual_body_treatment",
        "kind": "utility",
        "description": "Biography mentions hands-on physical treatment, spinal care, posture, or musculoskeletal therapy.",
        "example_terms": ["chiropractic", "spine", "manual therapy", "posture", "adjustment"],
    },
    {
        "concept": "classroom_instruction",
        "kind": "utility",
        "description": "Biography mentions classroom teaching, pupils, curriculum, school instruction, or students.",
        "example_terms": ["classroom", "students", "pupils", "curriculum", "school"],
    },
    {
        "concept": "university_affiliation",
        "kind": "utility",
        "description": "Biography mentions universities, departments, faculty roles, or academic appointments.",
        "example_terms": ["university", "faculty", "department", "professor", "college"],
    },
    {
        "concept": "research_publications",
        "kind": "utility",
        "description": "Biography mentions research, publications, grants, papers, journals, or scholarly work.",
        "example_terms": ["research", "publication", "journal", "paper", "grant"],
    },
    {
        "concept": "advanced_degree_or_training",
        "kind": "utility",
        "description": "Biography mentions graduate degrees, doctoral training, certifications, or formal advanced education.",
        "example_terms": ["phd", "doctoral", "master", "certified", "training"],
    },
    {
        "concept": "software_development",
        "kind": "utility",
        "description": "Biography mentions writing software, programming, development, open-source work, or engineering systems.",
        "example_terms": ["software", "developer", "programming", "github", "open source"],
    },
    {
        "concept": "technical_infrastructure",
        "kind": "utility",
        "description": "Biography mentions servers, databases, distributed systems, cloud tools, or technical platforms.",
        "example_terms": ["server", "database", "linux", "apache", "cloud"],
    },
    {
        "concept": "building_or_spatial_design",
        "kind": "utility",
        "description": "Biography mentions buildings, architecture, spatial planning, interiors, or construction projects.",
        "example_terms": ["architecture", "building", "design", "construction", "space"],
    },
    {
        "concept": "interior_or_decorative_design",
        "kind": "utility",
        "description": "Biography mentions interior spaces, furnishings, decor, residential design, or styling spaces.",
        "example_terms": ["interior", "furnishings", "decor", "residential", "space planning"],
    },
    {
        "concept": "visual_art_practice",
        "kind": "utility",
        "description": "Biography mentions painting, drawing, sculpture, galleries, exhibitions, or visual artwork.",
        "example_terms": ["painting", "canvas", "gallery", "exhibition", "artwork"],
    },
    {
        "concept": "camera_or_photography_work",
        "kind": "utility",
        "description": "Biography mentions photography, cameras, portraits, photo projects, or photographic work.",
        "example_terms": ["photography", "photograph", "camera", "portrait", "photo"],
    },
    {
        "concept": "film_or_video_production",
        "kind": "utility",
        "description": "Biography mentions film, directing, documentaries, screenings, festivals, or video production.",
        "example_terms": ["film", "documentary", "directing", "screened", "festival"],
    },
    {
        "concept": "reporting_or_news_work",
        "kind": "utility",
        "description": "Biography mentions reporting, journalism, newsrooms, correspondents, or editorial work.",
        "example_terms": ["reporter", "journalist", "news", "correspondent", "editor"],
    },
    {
        "concept": "literary_writing",
        "kind": "utility",
        "description": "Biography mentions poetry, books, literary publications, readings, or creative writing.",
        "example_terms": ["poetry", "poem", "book", "chapbook", "literary"],
    },
    {
        "concept": "music_composition",
        "kind": "utility",
        "description": "Biography mentions composing, scores, orchestras, music writing, or musical works.",
        "example_terms": ["composer", "score", "orchestra", "composition", "music"],
    },
    {
        "concept": "live_music_or_recording",
        "kind": "utility",
        "description": "Biography mentions live music, albums, tracks, recording, DJ work, or performance events.",
        "example_terms": ["album", "recording", "track", "dj", "performance"],
    },
    {
        "concept": "comedy_or_stage_performance",
        "kind": "utility",
        "description": "Biography mentions comedy, stand-up, stage performance, television appearances, or entertainment work.",
        "example_terms": ["comedy", "stand-up", "stage", "show", "performance"],
    },
    {
        "concept": "fashion_or_modeling_work",
        "kind": "utility",
        "description": "Biography mentions modeling, fashion, runway, campaigns, visual promotion, or appearances.",
        "example_terms": ["modeling", "fashion", "runway", "campaign", "swimsuit"],
    },
    {
        "concept": "fitness_instruction",
        "kind": "utility",
        "description": "Biography mentions exercise coaching, workouts, bodybuilding, strength training, or physical fitness.",
        "example_terms": ["fitness", "workout", "training", "bodybuilding", "exercise"],
    },
    {
        "concept": "mind_body_practice",
        "kind": "utility",
        "description": "Biography mentions yoga-style movement, breath work, postures, alignment, or relaxation practice.",
        "example_terms": ["yoga", "asana", "vinyasa", "breath", "alignment"],
    },
    {
        "concept": "religious_ministry",
        "kind": "utility",
        "description": "Biography mentions church work, ministry, worship, preaching, pastoral service, or congregations.",
        "example_terms": ["church", "ministry", "worship", "preaching", "congregation"],
    },
    {
        "concept": "public_speaking_or_events",
        "kind": "utility",
        "description": "Biography mentions talks, conferences, lectures, public events, workshops, or invited speaking.",
        "example_terms": ["speaker", "conference", "lecture", "workshop", "event"],
    },
    {
        "concept": "business_or_client_service",
        "kind": "utility",
        "description": "Biography mentions clients, consulting, business services, professional practice, or customer-facing work.",
        "example_terms": ["clients", "consulting", "business", "practice", "services"],
    },
    {
        "concept": "financial_or_accounting_work",
        "kind": "utility",
        "description": "Biography mentions accounting, taxes, payroll, auditing, finance, or bookkeeping.",
        "example_terms": ["accounting", "tax", "payroll", "audit", "bookkeeping"],
    },
    {
        "concept": "direct_gender_pronouns",
        "kind": "direct_sensitive",
        "description": "Biography uses gendered personal pronouns.",
        "example_terms": ["he", "him", "his", "she", "her", "hers"],
    },
    {
        "concept": "gendered_titles",
        "kind": "direct_sensitive",
        "description": "Biography uses explicitly gendered titles or honorifics.",
        "example_terms": ["mr", "mrs", "ms", "miss", "sir", "madam"],
    },
    {
        "concept": "gendered_family_roles",
        "kind": "proxy_sensitive",
        "description": "Biography mentions family roles that often reveal gender directly or indirectly.",
        "example_terms": ["wife", "husband", "mother", "father", "daughter", "son"],
    },
    {
        "concept": "parenting_or_caregiving_mentions",
        "kind": "proxy_sensitive",
        "description": "Biography mentions parenting, children, caregiving, or household family responsibilities.",
        "example_terms": ["children", "parent", "family", "caregiver", "home"],
    },
    {
        "concept": "gendered_name_signal",
        "kind": "proxy_sensitive",
        "description": "Biography contains first-name information that may strongly correlate with gender.",
        "example_terms": ["first name", "given name"],
    },
    {
        "concept": "appearance_or_body_presentation",
        "kind": "proxy_sensitive",
        "description": "Biography emphasizes appearance, beauty, body presentation, or visual attractiveness.",
        "example_terms": ["beauty", "appearance", "swimsuit", "glamour", "body"],
    },
    {
        "concept": "gendered_organization_or_award",
        "kind": "proxy_sensitive",
        "description": "Biography mentions gendered organizations, awards, lists, or groups.",
        "example_terms": ["women in", "female", "men's", "women's", "girls"],
    },
]

candidate_concepts_df = pd.DataFrame(candidate_concepts)

{
    "num_concepts": len(candidate_concepts_df),
    "by_kind": candidate_concepts_df["kind"].value_counts().to_dict(),
}

{'num_concepts': 40,
 'by_kind': {'utility': 33, 'proxy_sensitive': 5, 'direct_sensitive': 2}}

In [16]:
candidate_concepts_df[["concept", "kind", "description", "example_terms"]]

,concept,kind,description,example_terms
0,legal_practice,utility,"Biography mentions legal work, law practice, c...","[law, legal, court, litigation, counsel, case]"
1,legal_support_work,utility,Biography mentions support work around legal c...,"[paralegal, pleadings, filings, case preparati..."
2,clinical_care,utility,"Biography mentions direct clinical care, diagn...","[patient, clinic, treatment, diagnosis, health..."
3,medical_training,utility,"Biography mentions medical degrees, residency,...","[md, residency, fellowship, board certified, m..."
4,surgical_work,utility,"Biography mentions surgical procedures, operat...","[surgery, surgical, operation, procedure, orth..."
5,nursing_care,utility,"Biography mentions nursing care, nursing crede...","[nursing, registered nurse, nurse practitioner..."
6,dental_or_oral_health,utility,"Biography mentions dental care, oral health, t...","[dental, dentistry, oral health, crowns, dentu..."
7,therapy_or_counseling,utility,"Biography mentions therapy, counseling, psycho...","[therapy, counseling, psychotherapy, mental he..."
8,nutrition_or_diet_guidance,utility,"Biography mentions nutrition, diet planning, f...","[nutrition, diet, dietetic, meal planning, food]"
9,manual_body_treatment,utility,Biography mentions hands-on physical treatment...,"[chiropractic, spine, manual therapy, posture,..."


Before freezing this set, we do one lightweight validation pass:

- check that concepts are not almost always absent or almost always present
- check that no pair of concepts is nearly duplicate
- check that no utility concept behaves like a direct profession-label shortcut
- check that sensitive and proxy-sensitive concepts are clearly separated from utility concepts

This validation is only a sanity check. The goal is not to tune the concept set repeatedly, but to avoid obvious design mistakes before using it in the STAQ experiment.

## Lightweight Validation Pass

We now run one simple sanity check before freezing the concept set.

This validation uses the `example_terms` attached to each concept as a lightweight keyword approximation. It is not the final pseudo-labeling method. The goal is only to catch obvious concept-design mistakes: concepts that are nearly always absent, nearly duplicated, too close to one profession, or unexpectedly gender-revealing.

In [17]:
validation_sample_size = min(20000, len(raw_train))
validation_rows = raw_train.select(range(validation_sample_size))

validation_df = pd.DataFrame(
    {
        "profession": [profession_names[row[target_column]] for row in validation_rows],
        "gender": [gender_names[row[sensitive_column]] for row in validation_rows],
        "text": [row[text_column] for row in validation_rows],
    }
)


def term_to_regex(term):
    escaped = re.escape(term).replace(r"\ ", r"\s+")
    if term[0].isalnum():
        escaped = r"\b" + escaped
    if term[-1].isalnum():
        escaped = escaped + r"\b"
    return escaped


def concept_regex(example_terms):
    return re.compile("|".join(term_to_regex(term.lower()) for term in example_terms), flags=re.IGNORECASE)


concept_patterns = {
    row.concept: concept_regex(row.example_terms)
    for row in candidate_concepts_df.itertuples(index=False)
}

concept_hits = pd.DataFrame(
    {
        concept: validation_df["text"].str.contains(pattern, regex=True, na=False)
        for concept, pattern in concept_patterns.items()
    }
)

{
    "validation_examples": len(validation_df),
    "num_concepts": concept_hits.shape[1],
}

{'validation_examples': 20000, 'num_concepts': 40}

### Coverage

A concept is suspicious if the lightweight keyword check almost never fires, because it may be hard to label reliably. A concept is also suspicious if it fires on most biographies, because it may be too generic to be a useful query.

These thresholds are intentionally loose. Rare concepts are not automatically bad, but they deserve inspection.

In [18]:
coverage_df = candidate_concepts_df[["concept", "kind", "description"]].copy()
coverage_df["hit_count"] = [int(concept_hits[concept].sum()) for concept in coverage_df["concept"]]
coverage_df["hit_rate"] = coverage_df["hit_count"] / len(validation_df)

coverage_df["coverage_flag"] = "ok"
coverage_df.loc[coverage_df["hit_rate"] < 0.01, "coverage_flag"] = "very_rare"
coverage_df.loc[coverage_df["hit_rate"] > 0.60, "coverage_flag"] = "very_common"

coverage_df.sort_values(["coverage_flag", "hit_rate", "concept"], ascending=[True, True, True])

,concept,kind,description,hit_count,hit_rate,coverage_flag
29,religious_ministry,utility,"Biography mentions church work, ministry, wors...",207,0.01035,ok
39,gendered_organization_or_award,proxy_sensitive,"Biography mentions gendered organizations, awa...",311,0.01555,ok
32,financial_or_accounting_work,utility,"Biography mentions accounting, taxes, payroll,...",324,0.01620,ok
38,appearance_or_body_presentation,proxy_sensitive,"Biography emphasizes appearance, beauty, body ...",381,0.01905,ok
8,nutrition_or_diet_guidance,utility,"Biography mentions nutrition, diet planning, f...",402,0.02010,ok
26,fashion_or_modeling_work,utility,"Biography mentions modeling, fashion, runway, ...",416,0.02080,ok
30,public_speaking_or_events,utility,"Biography mentions talks, conferences, lecture...",450,0.02250,ok
24,live_music_or_recording,utility,"Biography mentions live music, albums, tracks,...",469,0.02345,ok
14,software_development,utility,"Biography mentions writing software, programmi...",477,0.02385,ok
18,visual_art_practice,utility,"Biography mentions painting, drawing, sculptur...",486,0.02430,ok


### Overlap

Next we check whether two concepts behave like near-duplicates under the keyword approximation.

The most useful columns are `jaccard` and the two conditional overlaps. High conditional overlap means that whenever one concept appears, the other almost always appears too.

In [19]:
overlap_rows = []
concept_names = list(concept_hits.columns)

for left_idx, left in enumerate(concept_names):
    left_hits = concept_hits[left]
    left_count = int(left_hits.sum())
    if left_count == 0:
        continue

    for right in concept_names[left_idx + 1 :]:
        right_hits = concept_hits[right]
        right_count = int(right_hits.sum())
        if right_count == 0:
            continue

        both_count = int((left_hits & right_hits).sum())
        either_count = int((left_hits | right_hits).sum())
        if both_count == 0:
            continue

        overlap_rows.append(
            {
                "left": left,
                "right": right,
                "left_kind": candidate_concepts_df.set_index("concept").loc[left, "kind"],
                "right_kind": candidate_concepts_df.set_index("concept").loc[right, "kind"],
                "both_count": both_count,
                "jaccard": both_count / either_count,
                "p_right_given_left": both_count / left_count,
                "p_left_given_right": both_count / right_count,
            }
        )

overlap_df = pd.DataFrame(overlap_rows)

near_duplicate_df = overlap_df[
    (overlap_df["jaccard"] >= 0.25)
    | (overlap_df[["p_right_given_left", "p_left_given_right"]].max(axis=1) >= 0.75)
].sort_values(
    ["jaccard", "p_right_given_left", "p_left_given_right"],
    ascending=False,
)

near_duplicate_df.head(25)

,left,right,left_kind,right_kind,both_count,jaccard,p_right_given_left,p_left_given_right
351,university_affiliation,direct_gender_pronouns,utility,direct_sensitive,7760,0.407627,0.987026,0.409823
595,live_music_or_recording,comedy_or_stage_performance,utility,utility,310,0.361727,0.660981,0.444126
396,advanced_degree_or_training,fitness_instruction,utility,utility,1050,0.312407,0.322779,0.906736
302,classroom_instruction,university_affiliation,utility,utility,2537,0.285024,0.709452,0.322691
331,university_affiliation,advanced_degree_or_training,utility,utility,2380,0.272467,0.302722,0.731632
330,university_affiliation,research_publications,utility,utility,2611,0.265724,0.332104,0.570710
377,research_publications,direct_gender_pronouns,utility,direct_sensitive,4560,0.240633,0.996721,0.240824
324,classroom_instruction,direct_gender_pronouns,utility,direct_sensitive,3544,0.186851,0.991051,0.187167
102,medical_training,university_affiliation,utility,utility,1436,0.174951,0.805836,0.182651
673,business_or_client_service,direct_gender_pronouns,utility,direct_sensitive,3350,0.174452,0.925926,0.176921


### Profession Shortcut Risk

A utility concept should provide evidence, not behave like a direct profession label.

For each utility concept, we look at the most common profession among matching biographies. A high `top_profession_share` means the concept is concentrated in one class. This is not always wrong, but it is the main shortcut risk to inspect.

In [20]:
base_profession_rates = validation_df["profession"].value_counts(normalize=True)
utility_concepts = candidate_concepts_df.query("kind == 'utility'")["concept"].tolist()

shortcut_rows = []
for concept in utility_concepts:
    hits = concept_hits[concept]
    hit_count = int(hits.sum())
    if hit_count == 0:
        shortcut_rows.append(
            {
                "concept": concept,
                "hit_count": 0,
                "top_profession": None,
                "top_profession_share": 0.0,
                "top_profession_base_rate": 0.0,
                "lift_over_base": 0.0,
                "shortcut_flag": "no_hits",
            }
        )
        continue

    profession_rates = validation_df.loc[hits, "profession"].value_counts(normalize=True)
    top_profession = profession_rates.index[0]
    top_share = float(profession_rates.iloc[0])
    base_rate = float(base_profession_rates[top_profession])

    shortcut_rows.append(
        {
            "concept": concept,
            "hit_count": hit_count,
            "top_profession": top_profession,
            "top_profession_share": top_share,
            "top_profession_base_rate": base_rate,
            "lift_over_base": top_share / base_rate,
            "shortcut_flag": "inspect" if hit_count >= 20 and top_share >= 0.65 else "ok",
        }
    )

shortcut_df = pd.DataFrame(shortcut_rows).sort_values(
    ["shortcut_flag", "top_profession_share", "hit_count"],
    ascending=[True, False, False],
)

shortcut_df

,concept,hit_count,top_profession,top_profession_share,top_profession_base_rate,lift_over_base,shortcut_flag
6,dental_or_oral_health,641,dentist,0.862715,0.03550,24.301817,inspect
5,nursing_care,757,nurse,0.850727,0.04815,17.668257,inspect
19,camera_or_photography_work,890,photographer,0.822472,0.06385,12.881314,inspect
12,research_publications,4575,professor,0.806776,0.30195,2.671886,inspect
0,legal_practice,1806,attorney,0.671096,0.08075,8.310791,inspect
1,legal_support_work,63,paralegal,0.587302,0.00485,121.093111,ok
32,financial_or_accounting_work,324,accountant,0.537037,0.01490,36.042754,ok
13,advanced_degree_or_training,3253,professor,0.507224,0.30195,1.679828,ok
28,mind_body_practice,165,yoga_teacher,0.466667,0.00455,102.564103,ok
11,university_affiliation,7862,professor,0.455100,0.30195,1.507205,ok


### Sensitive Separation

Finally, we check how strongly each concept differs by the dataset gender label.

Direct-sensitive and proxy-sensitive concepts are expected to show gender gaps. Utility concepts with large gaps are not automatically invalid, but they should be reviewed because they may behave like sensitive proxies.

In [21]:
gender_values = list(validation_df["gender"].dropna().unique())

sensitive_rows = []
for concept in concept_names:
    row = {
        "concept": concept,
        "kind": candidate_concepts_df.set_index("concept").loc[concept, "kind"],
        "overall_hit_rate": float(concept_hits[concept].mean()),
    }

    gender_rates = []
    for gender in gender_values:
        mask = validation_df["gender"] == gender
        rate = float(concept_hits.loc[mask, concept].mean())
        row[f"hit_rate_{gender}"] = rate
        gender_rates.append(rate)

    row["max_gender_gap"] = max(gender_rates) - min(gender_rates)
    row["sensitive_review_flag"] = (
        "inspect_utility_proxy"
        if row["kind"] == "utility" and row["max_gender_gap"] >= 0.10
        else "expected_or_ok"
    )
    sensitive_rows.append(row)

sensitive_gap_df = pd.DataFrame(sensitive_rows).sort_values(
    ["sensitive_review_flag", "max_gender_gap", "overall_hit_rate"],
    ascending=[True, False, False],
)

sensitive_gap_df

,concept,kind,overall_hit_rate,hit_rate_male,hit_rate_female,max_gender_gap,sensitive_review_flag
36,parenting_or_caregiving_mentions,proxy_sensitive,0.10305,0.071071,0.140425,0.069354,expected_or_ok
5,nursing_care,utility,0.03785,0.008907,0.071676,0.062769,expected_or_ok
34,gendered_titles,direct_sensitive,0.08015,0.052514,0.112448,0.059934,expected_or_ok
33,direct_gender_pronouns,direct_sensitive,0.94675,0.969382,0.920299,0.049083,expected_or_ok
22,literary_writing,utility,0.10045,0.077844,0.126871,0.049027,expected_or_ok
3,medical_training,utility,0.08910,0.105307,0.070158,0.035149,expected_or_ok
11,university_affiliation,utility,0.39310,0.407868,0.375840,0.032027,expected_or_ok
4,surgical_work,utility,0.04660,0.060215,0.030687,0.029528,expected_or_ok
14,software_development,utility,0.02385,0.034793,0.011061,0.023733,expected_or_ok
16,building_or_spatial_design,utility,0.06590,0.076081,0.054001,0.022080,expected_or_ok


### Freeze Decision

Use the tables above as a final sanity check, not as an invitation to repeatedly tune the concept set.

A reasonable freeze criterion is:

- no concept is broken by construction, such as zero hits from obvious spelling problems
- no two utility concepts are near-duplicates
- no utility concept is so concentrated that it effectively names one profession
- concepts with large gender gaps are either intentionally sensitive/proxy-sensitive or are explicitly reviewed

If these checks look acceptable, the proposed concept set is good enough to freeze for the BIOS STAQ experiment.